# Stochastic Calculus from Zero · Episode 20: Simulating SDEs: Euler versus Milstein

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The share** (Series 3's): priced at 100, 20% volatility, a 3.60% risk-free rate (continuous), a call with strike 105 expiring in a year, and an illustrative real-world drift of 8%. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. Euler · 4. Milstein · 5. Strong error: paths · 6. Weak error: prices · 7. Which to use · 8. This episode's question · 9. Exercises · 10. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 20,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))

## 2. Last episode's answer

$\sigma = 0.1$: $\sigma^2 = 0.01 < 2a\theta = 0.021$. Feller's condition holds.

In [ ]:
out["prev"] = {"answer": "yes"}

## 3. Euler

Most SDEs have no formula, so we step them. Freeze the drift and volatility at the start of each step:

$$X_{n+1} = X_n + \mu(X_n)\,\Delta t + \sigma(X_n)\,\Delta W_n .$$

This is the **Euler** (Euler-Maruyama) scheme. Its error comes from treating $\sigma(X_s)$ as constant over the step, when $X$ moves by about $\sigma\,\Delta W$, which is of order $\sqrt{\Delta t}$.

## 4. Milstein

Expand the volatility over the step: $\sigma(X_s) \approx \sigma(X_n) + \sigma'(X_n)\,\sigma(X_n)\,(W_s - W_n)$. The extra piece integrates to

$$\int_{t_n}^{t_{n+1}} (W_s - W_n)\,dW_s = \tfrac12\big((\Delta W_n)^2 - \Delta t\big)$$

(Part 2, Episode 13: the Itô integral of $W\,dW$). So the **Milstein** scheme is

$$X_{n+1} = X_n + \mu\,\Delta t + \sigma\,\Delta W_n + \tfrac12\,\sigma\sigma'\big((\Delta W_n)^2 - \Delta t\big).$$

For GBM, $\sigma(x) = \sigma x$ and $\sigma\sigma' = \sigma^2 x$.

In [ ]:
S0, VOL, MU = 100.0, 0.20, 0.08
out["history"] = {"milstein": 1974}
rng = np.random.default_rng(seed)
P, NFINE = 5_000, 1024
Z = rng.standard_normal((P, NFINE)) * math.sqrt(1 / NFINE)       # one set of Brownian steps on the finest grid
exact = S0 * np.exp((MU - 0.5 * VOL ** 2) + VOL * Z.sum(axis=1))  # GBM's exact solution on the same draws

def schemes(n):
    '''Euler and Milstein with n steps, driven by the fine steps added up in blocks.'''
    dW = Z.reshape(P, n, NFINE // n).sum(axis=2)
    dt = 1 / n
    e = np.full(P, S0); m = np.full(P, S0)
    for i in range(n):
        e = e + MU * e * dt + VOL * e * dW[:, i]
        m = m + MU * m * dt + VOL * m * dW[:, i] + 0.5 * VOL ** 2 * m * (dW[:, i] ** 2 - dt)
    return e, m

## 5. Strong error: paths

**Strong error** asks how far each simulated path ends from the true one, on the same random draws: $E|X_T^{\text{scheme}} - X_T|$. Theory: Euler's falls like $\Delta t^{1/2}$, Milstein's like $\Delta t$.

In [ ]:
rows = []
for n in (4, 16, 64, 256, 1024):
    e, m = schemes(n)
    rows.append({"steps": n, "euler_strong": float(np.abs(e - exact).mean()), "milstein_strong": float(np.abs(m - exact).mean()),
                 # weak error on the same draws as the exact solution, so sampling noise cancels
                 "euler_weak": float(abs(e.mean() - exact.mean())), "milstein_weak": float(abs(m.mean() - exact.mean()))})
x = np.log([r["steps"] for r in rows])
slope = lambda key: float(np.polyfit(x, np.log([r[key] for r in rows]), 1)[0])
conv = {"rows": rows, "paths": P, "slope_euler": slope("euler_strong"), "slope_milstein": slope("milstein_strong"),
        "slope_euler_weak": slope("euler_weak"), "slope_milstein_weak": slope("milstein_weak"),
        "x": list(range(1, 6)), "euler": [r["euler_strong"] for r in rows], "milstein": [r["milstein_strong"] for r in rows],
        "ticks": [{"x": i + 1, "label": f"{r['steps']:,}"} for i, r in enumerate(rows)],
        "first": rows[0], "last": rows[-1], "ratio_last": rows[-1]["euler_strong"] / rows[-1]["milstein_strong"]}
assert -0.6 < conv["slope_euler"] < -0.4 and -1.1 < conv["slope_milstein"] < -0.9
out["conv"] = conv
pd.DataFrame(rows)

## 6. Weak error: prices

**Weak error** asks only about averages: $|E[X_T^{\text{scheme}}] - E[X_T]|$, which is what a price needs. For Euler on GBM, $E[S_N] = S_0(1 + \mu\Delta t)^N$ against $S_0e^{\mu T}$: an error of order $\Delta t$. Milstein's extra term has mean zero, so it doesn't help: both schemes are **weak order 1**. (The table measures the weak error on the same draws as the exact solution, so sampling noise cancels.)

In [ ]:
weak_theory = [S0 * abs((1 + MU / r["steps"]) ** r["steps"] - math.exp(MU)) for r in rows]
conv["weak_theory"] = weak_theory
for r, w in zip(rows, weak_theory):
    r["euler_weak_theory"] = w
pd.DataFrame(rows)[["steps", "euler_weak", "euler_weak_theory", "milstein_weak"]]

## 7. Which to use

- Path-by-path accuracy (hedging simulations, path-dependent payoffs, multilevel Monte Carlo): Milstein, or an exact scheme where one exists (GBM, Vasicek).
- Prices of plain payoffs: the scheme's bias is usually smaller than the Monte Carlo noise; Euler with more paths is fine.
- In several dimensions Milstein needs extra terms (iterated integrals of different Brownian motions), which is why Euler stays popular.

## 8. This episode's question

Euler's strong error falls like $\sqrt{\Delta t}$, so halving it needs 4 times as many steps: 4,096.

In [ ]:
out["question"] = {"steps": 1024, "error": conv["last"]["euler_strong"], "answer_steps": 4096}

## 9. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Write the Milstein scheme for GBM, $dS = \mu S\,dt + \sigma S\,dW$.

<details><summary>Answer</summary>

$\sigma(x) = \sigma x$, $\sigma'(x) = \sigma$: $S_{n+1} = S_n\big(1 + \mu\Delta t + \sigma\Delta W + \tfrac12\sigma^2((\Delta W)^2 - \Delta t)\big)$.

</details>

**Exercise 2 (By hand).** For $dX = \mu\,dt + \sigma\,dW$ with constants, what does Milstein add to Euler?

<details><summary>Answer</summary>

Nothing: $\sigma' = 0$. Euler is then exact on the grid points, for both strong and weak error.

</details>

**Exercise 3 (By hand).** Why is Euler's strong order only $\tfrac12$? (Hint: the size of the term it drops.)

<details><summary>Answer</summary>

It drops $\tfrac12\sigma\sigma'((\Delta W)^2 - \Delta t)$, of size $\Delta t$ per step. These errors have mean zero and are independent, so over $N = T/\Delta t$ steps they add up like $\sqrt{N}\,\Delta t = \sqrt{T\Delta t}$.

</details>

**Exercise 4 (In code).** Apply both schemes to the CIR model of Episode 19 ($\sigma(r) = \sigma\sqrt r$, so $\sigma\sigma' = \sigma^2/2$) with 12 steps a year for 5 years. How often does each go negative?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
R0, THETA, A, SIG = 0.0385, 0.035, 0.30, 0.25        # Feller broken, to make negatives possible
rng_c = np.random.default_rng(4)
n, dt = 50_000, 1 / 12
e = np.full(n, R0); m = np.full(n, R0); neg_e = np.zeros(n, bool); neg_m = np.zeros(n, bool)
for _ in range(60):
    dW = math.sqrt(dt) * rng_c.standard_normal(n)
    ep, mp = np.maximum(e, 0), np.maximum(m, 0)
    e = e + A * (THETA - ep) * dt + SIG * np.sqrt(ep) * dW
    m = m + A * (THETA - mp) * dt + SIG * np.sqrt(mp) * dW + 0.25 * SIG ** 2 * (dW ** 2 - dt)
    neg_e |= e < 0; neg_m |= m < 0
assert neg_m.mean() < neg_e.mean()
print(f"went negative: Euler {neg_e.mean():.1%}, Milstein {neg_m.mean():.1%}")

**Exercise 5 (In code).** Price the call (strike 105) under the risk-neutral drift 3.6% with Euler at 4, 16 and 64 steps, on the same draws as the exact solution. How fast does the price error fall?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
R, K = 0.036, 105.0
ex = S0 * np.exp((R - 0.5 * VOL ** 2) + VOL * Z.sum(axis=1))
exact_price = math.exp(-R) * np.maximum(ex - K, 0).mean()
errs = []
for n in (4, 16, 64):
    dW = Z.reshape(P, n, NFINE // n).sum(axis=2)
    e = np.full(P, S0)
    for i in range(n):
        e = e + R * e / n + VOL * e * dW[:, i]
    errs.append(abs(math.exp(-R) * np.maximum(e - K, 0).mean() - exact_price))
assert errs[-1] < errs[0]
print("price error:", [round(float(x), 4) for x in errs], "(it shrinks as the steps grow)")

## 10. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")